In [1]:
import glob

data_dir = 'data/'

def get_data_files():
    # list all folders in data directory
    folders = glob.glob(data_dir + '*/')

    data = {}
    # list all files in each folder
    for folder in folders:
        files = glob.glob(folder + '*')
        data[folder[4:].strip('\\')] = [file.replace('\\', '/') for file in files]

    return data

In [2]:
data = get_data_files()
target = data['Target']
walmart = data['Walmart']
kmart = data['kmart']

In [3]:
import pandas as pd
import altair as alt
from vega_datasets import data

# Enable data transformer for larger datasets
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [4]:
target_locations = pd.read_csv(target[1])[['geolocation_lat', 'geolocation_lng', 'geolocation_city']].rename(columns={'geolocation_lat': 'lat', 'geolocation_lng': 'lon', 'geolocation_city': 'city'})
# Preprocess count orders by a promixity of 0.1 degree 6.5 miles range
target_locations['lat'] = target_locations['lat'].apply(lambda x: round(x,1))
target_locations['lon'] = target_locations['lon'].apply(lambda x: round(x, 1))
target_locations['count'] = 1   
target_locations = target_locations.groupby(['lat', 'lon', 'city']).count().reset_index()
target_locations.head()

,lat,lon,city,count
0,-36.6,-64.3,santa rosa,3
1,-34.6,-58.9,itabata,1
2,-34.6,-58.7,santa maria,1
3,-33.7,-53.5,chui,41
4,-33.7,-53.5,chuí,15


In [5]:
# Define Brazil's latitude and longitude boundaries
lat_min = -33.75  # Southernmost point
lat_max = -5.27   # Northernmost point (since it's south of the Equator, it's negative)
lon_min = -73.99  # Westernmost point
lon_max = -34.79  # Easternmost point

# Filter outliers (orders outside Brazil)
outliers = target_locations[
    (target_locations['lat'] < lat_min) | (target_locations['lat'] > lat_max) |
    (target_locations['lon'] < lon_min) | (target_locations['lon'] > lon_max)
]

# Keep only orders within Brazil
target_locations = target_locations[
    (target_locations['lat'] >= lat_min) & (target_locations['lat'] <= lat_max) &
    (target_locations['lon'] >= lon_min) & (target_locations['lon'] <= lon_max)
]


In [12]:
# Get map of Brazil
countries = alt.topo_feature(data.world_110m.url, 'countries')
SA_CODES = [32, 68, 76, 152, 170, 218, 328, 600, 604, 740, 858, 862]

# Create a map of Brazil
background = alt.Chart(countries).mark_geoshape(
    stroke='black'
).transform_filter(
    alt.FieldOneOfPredicate(field='id', oneOf=SA_CODES)
).properties(
    width=1000,
    height=600
)

# Plot points
points = alt.Chart(target_locations).mark_circle().encode(
    longitude='lon:Q',
    latitude='lat:Q',
    tooltip=[
        alt.Tooltip('city:N', title='City'),
        alt.Tooltip('count:Q', title='Original Count'),
    ],
    size=alt.value(2),
    # color by count
    color=alt.Color('count:Q', title="Log Scaled Count", scale=alt.Scale(scheme='spectral', type='log')),
).properties(
    title='Target Locations',
    width=1000,
    height=600
).project(
    type='mercator'
)

background + points

alt.LayerChart(...)

## Why this Visualization?
I wanted to visualize the distribution of the target order locations in brazil on a map that can accurately represent the distribution, and make it easy to discern the hotspots and coldspots. 

I chose this visualization because
- This visualization is a simple and effective way to understand the distribution of the data.
- It is easy to understand and interpret.
- It is visually appealing and can be used to present the data to a non-technical audience.

**Bonus points are tooltips, that show the original count**

**It is also in a log scaling due to the large range of the data.**